In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# cls_parser.pkl
COPY passwords.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
snowflake-connector-python

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from passwords import *
import snowflake.connector
from datetime import datetime

# constants
str_project = '20231010-gen-xii'
str_task = 'ad_hoc'
str_subtask = 'gen_11_payload_parsing'
str_final_task = 'payloads'

# get today's date
str_date_today = datetime.today().strftime('%Y%m%d')

# connect to snowflake
conn = snowflake.connector.connect(
    user=USERNAME,
    password=PASSWORD,
    account=ACCOUNT,
    warehouse=WAREHOUSE,
    database=DATABASE,
    schema=SCHEMA,
)

# query
str_query = """
select *
from 
raw.source_s3_scorehistory.PAYLOADS_PARSED_YESTERDAY
"""

# pull payloads
df = pd.read_sql(
    sql=str_query,
    con=conn,
)

# subset to gen 12
df = df[df['RESPONSE_MODEL_NAME'] == 'PRESTIGE-GENXI'].copy()

# write to s3
str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/days/{str_date_today}/{str_final_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxi-pull-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  18.43kB
Step 1/8 : FROM python:3.9
3.9: Pulling from library/python
1468e7ff95fc: Pulling fs layer
2cf9c2b42f41: Pulling fs layer
c4c40c3e3cdf: Pulling fs layer
c05cc1123d7e: Pulling fs layer
b6f29ccdcc55: Pulling fs layer
9c8be2164d2a: Pulling fs layer
d00b60686603: Pulling fs layer
1320d210901a: Pulling fs layer
b6f29ccdcc55: Waiting
9c8be2164d2a: Waiting
d00b60686603: Waiting
1320d210901a: Waiting
c05cc1123d7e: Waiting
2cf9c2b42f41: Download complete
1468e7ff95fc: Verifying Checksum
1468e7ff95fc: Download complete
c4c40c3e3cdf: Verifying Checksum
c4c40c3e3cdf: Download complete
b6f29ccdcc55: Verifying Checksum
b6f29ccdcc55: Download complete
9c8be2164d2a: Verifying Checksum
9c8be2164d2a: Download complete
d00b60686603: Verifying Checksum
d00b60686603: Download complete
1320d210901a: Download complete
1468e7ff95fc: Pull complete
c05cc1123d7e: Download complete
2cf9c2b42f41: Pull complete
c4c40c3e3cdf: Pull complete
c05cc1123d7e: Pull complete
b

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.4/443.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-west-2:836690756591:repository/genxi-pull-payloads",
        "registryId": "836690756591",
        "repositoryName": "genxi-pull-payloads",
        "repositoryUri": "836690756591.dkr.ecr.us-west-2.amazonaws.com/genxi-pull-payloads",
        "createdAt": 1714497057.504,
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": true
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxi-pull-payloads]
f03e4df6638d: Preparing
1365c2d15fbb: Preparing
09695c4dacee: Preparing
39870207ce83: Preparing
1fbb04935245: Preparing
126ef28403f3: Preparing
47e31a4d606a: Preparing
bf4966b4b813: Preparing
da15a2a37253: Preparing
89ca33c95b2e: Preparing
83db175c22e2: Preparing
c5d13b2949a2: Preparing
7e43f593c900: Preparing
126ef28403f3: Waiting
072

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass